# Prepare data for the three selected models
**Goal:** create one external prepared-data folder that logistic regression, histogram gradient boosting and TabM consume through the same preparation loader. Run top to bottom from the repository root or `notebooks/`.

Dataset findings belong in the data-insights notebook and document. This notebook prepares inputs; it does not train or score classifiers.

```text
DATA FLOW
Local ECGDeli features + PTB-XL annotations
                    |
            validate and join by ECG ID
                    |
       select 504 lead fields + separate labels
                    |
         external prepared folder (five files)
                    |
      shared loader selects a patient split
                    |
        ready numerical X, y and weights
                    |
   logistic regression | histogram boosting | TabM

PROCESS FLOW
Validate -> group patients -> save package -> verify model input
                            loader: construct features
                                 -> fit training-only settings
                                 -> transform train and held-out data
```

## 1. Locate local inputs
Install `requirements.txt` first. Required source files are `features/ecgdeli_features.csv`, its sibling `labels/ptbxl_statements.csv`, and `reference/ptbxl_database.csv` under the external raw-data folder. The repository README gives download instructions. Sources are [PTB-XL+ 1.0.1](https://physionet.org/content/ptb-xl-plus/1.0.1/) and [PTB-XL 1.0.3](https://physionet.org/content/ptb-xl/1.0.3/).

In [ ]:
from pathlib import Path
import sys
import numpy as np

PROJECT = Path.cwd().resolve()
if PROJECT.name == 'notebooks':
    PROJECT = PROJECT.parent
if not (PROJECT / 'src/afib/data_preparation.py').is_file():
    raise FileNotFoundError('Run from the repository root or notebooks folder.')
sys.dont_write_bytecode = True
sys.path.insert(0, str(PROJECT / 'src'))
from afib.data_preparation import prepare_dataset, load_processed_data, load_prepared_split
from afib.paths import processed_directory
OUTPUT = processed_directory(PROJECT)

## 2. Validate, select and split
`prepare_dataset` performs these steps in order:

1. Check unique ECG IDs, matching source records, valid patient IDs and agreement between AFIB annotations. AFIB means code membership, including likelihood zero; absence does not mean healthy.
2. Select the **504 lead fields** (480 numerical and 24 categorical morphology fields). Exclude the 27 Global fields and all IDs, diagnoses and quality flags from predictors. Retain rows, natural gaps, meaningful zeros and unusual values; convert infinities to missing in the modelling copy. Original source files stay outside the repository in `AFIB_DATA_DIR` (default `../afib_data/raw`).
3. Assign **10 outer and 3 inner folds manually by patient**, stratifying on whether a patient has any AFIB-labelled recording. A patient's recordings remain together; inner folds use only outer-training patients.
4. Write the five-file package below. No imputation, scaling or category vocabulary is learned across the complete cohort.

In [ ]:
manifest = prepare_dataset(PROJECT)
print(f"Prepared {manifest['rows']:,} ECGs from {manifest['patients']:,} patients.")
print(f'External output folder: {OUTPUT}')
print('Files:', ', '.join(sorted(path.name for path in OUTPUT.iterdir())))

## 3. Check the saved package

| File in the external prepared folder | Purpose |
| --- | --- |
| `lead_features.npz` | 504 selected lead fields, original missingness, ECG IDs and column names |
| `targets.csv` | ECG ID, patient ID and binary `target_afib` |
| `folds.csv` | The same identifiers/target, `outer_fold` and ten `inner_fold_when_outer_*` columns; -1 marks that outer test set |
| `metadata.csv` | Four source quality flags, reserved for later subgroup reporting |
| `manifest.json` | Source/package hashes, schema, seed and preparation settings |

The shared loader checks file hashes, record alignment and patient grouping before use. Missing values are intentionally retained here so each selected training split can learn its own preparation.

In [ ]:
package = load_processed_data(OUTPUT)
assert package['raw'].shape == (manifest['rows'], 504)
assert package['folds'].groupby('patient_id').outer_fold.nunique().eq(1).all()
print('Saved files, feature order and patient folds verified.')

## 4. Produce ready-to-fit model inputs
`load_prepared_split` is the final preprocessing step shared by every algorithm. It selects one partition, constructs the requested features, fits numerical medians/means/scales and morphology categories **only on its observed training records**, and applies that mapping to held-out records. All-missing numerical training fields use zero; zero scales use one. Numerical missingness flags and explicit missing/unknown categories remain available.

| Arm | Features before encoding | Training rows |
| --- | --- | --- |
| A | Original 504 | Observed |
| B | Original + 63 row-wise cross-lead features | Observed |
| C | Original 504 | Observed + one corrupted copy |
| D | Original + 63 row-wise cross-lead features | Observed + one corrupted copy |

Constructed features use available measured values before filling and are rebuilt after any corruption. C/D split each ECG's class-balanced weight equally across its two views. Training corruption uses a separate random stream. Pass the returned `sample_weight` to the model; do not add model class weighting.

Choose `inner_fold=0`, `1` or `2` for tuning. Use `inner_fold=None` only for the outer refit/test stage. The example below verifies one inner split in memory and writes no additional files.

In [ ]:
prepared = load_prepared_split(OUTPUT, outer_fold=0, inner_fold=0, arm='A')
assert np.isfinite(prepared['X_train'].to_numpy()).all()
assert np.isfinite(prepared['X_valid'].to_numpy()).all()
assert prepared['X_train'].columns.equals(prepared['X_valid'].columns)
assert set(prepared['train_metadata'].patient_id).isdisjoint(prepared['valid_metadata'].patient_id)
print('Training matrix:', prepared['X_train'].shape)
print('Validation matrix:', prepared['X_valid'].shape)
print('Ready for the selected model: X_train, y_train, sample_weight, X_valid and y_valid.')

## Handoff
Each folder in `models/` uses `AFIB_PROCESSED_DIR` (default `../afib_data/processed`) and this same loader. Model fitting, parameter selection, stress-test scoring and result plots belong in the algorithm/evaluation stage. The project plan defines those experiments. Keep raw and prepared datasets outside the submitted repository. The equivalent terminal command is `python scripts/prepare_data.py`.